In [13]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [14]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [15]:
v1
len(v1)

384

In [16]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [17]:
dv.shape

(384,)

In [18]:
v1.dot(dv)

np.float32(0.32332397)

In [19]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [20]:
v2.dot(dv)

np.float32(0.019730438)

In [21]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-28 17:44:34--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 888 [text/plain]
Saving to: ‘ingest.py.1’

ingest.py.1         100%[===================>]     888  --.-KB/s    in 0s      

2026-06-28 17:44:34 (48.3 MB/s) - ‘ingest.py.1’ saved [888/888]



In [22]:
from ingest import load_faq_data

documents = load_faq_data()

In [23]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [24]:
texts[10]

'Course: How many hours per week am I expected to spend on this course? It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'

In [25]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [26]:
texts[5:10]
vectors[5:10]

[array([ 3.71673075e-03, -7.74852037e-02,  1.84528325e-02,  2.00679600e-02,
         3.45686600e-02,  5.21531561e-03, -5.73153235e-02, -8.92564729e-02,
        -4.59613428e-02, -7.94031937e-03,  3.75875644e-02,  8.92491953e-04,
         2.30120625e-02, -1.86401028e-02, -1.99409481e-02, -4.27778438e-02,
        -7.19160959e-02, -7.36176670e-02,  4.62202169e-02, -1.70772746e-02,
         2.77097449e-02, -5.81056178e-02, -4.60712463e-02,  3.63632441e-02,
         2.32860446e-02, -6.92727184e-03, -2.27100844e-03,  2.77587231e-02,
         4.36605662e-02, -1.03752799e-02,  3.60393412e-02,  8.94796997e-02,
         6.51182383e-02,  1.09147076e-02,  2.38105301e-02, -1.02274669e-02,
         2.49289684e-02, -1.11511657e-02, -6.83785230e-02,  5.90060987e-02,
        -1.62854709e-03, -1.93510316e-02,  2.03343760e-02,  5.42354248e-02,
         1.81330983e-02,  4.24618414e-03, -4.36290279e-02, -1.08390495e-01,
        -5.67716099e-02,  1.14050061e-01, -3.29341218e-02, -4.91560437e-02,
        -2.0

In [27]:
import numpy as np
X = np.array(vectors)

In [28]:
X.shape

(1350, 384)

In [29]:
scores = X.dot(v1)

In [30]:
scores

array([ 0.48740575,  0.20991933,  0.762941  , ..., -0.08637968,
        0.03759793, -0.03037044], shape=(1350,), dtype=float32)

In [31]:
idx = np.argmax(scores)
idx, scores[idx], documents[idx]

(np.int64(2),
 np.float32(0.762941),
 {'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: Can I still join the course after the start date?',
  'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
  'doc_id': '3f1424af17'})

In [32]:
top5 = np.argsort(-scores)[:5]

In [33]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579371
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192132
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 

In [34]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [35]:
results = vindex.search(v1, num_results=5, filter_dict={"course": "llm-zoomcamp"})
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.',
  'doc_id': 'bd31146b0e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed an

In [36]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-28 17:46:14--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.1’

rag_helper.py.1     100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-28 17:46:14 (28.3 MB/s) - ‘rag_helper.py.1’ saved [2134/2134]



In [37]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [38]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)
index.fit(documents)

In [39]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [40]:
query = 'I just found out about the program, can I still enroll?'
assistant.rag(query)

'Yes, you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still open.'

In [ ]:

class RAGVector(RAGBase):  

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [42]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [43]:
query = 'I just found out about the program, can I still sign up?'
vector_assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while submissions are still open.'

In [46]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [47]:
vs_index.fit(vectors, documents)

In [50]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5, filter_dict={"course": "llm-zoomcamp"})

In [51]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere

In [52]:
vs_index.close()